# 05 – Exploratory Data Analysis with SQL

**Goal:** answer 10 business questions with SQL.

**Data:** `Spacex.csv` from the IBM course (one row per Falcon 9 launch, 2010–2020, built from Wikipedia).
We load it into an **SQLite** database (built into Python) and run every query with `pandas.read_sql`.

In [1]:
import os, sqlite3
import pandas as pd

CSV = "../data/Spacex.csv"
if not os.path.exists(CSV):   # download the course file once
    url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
    pd.read_csv(url).to_csv(CSV, index=False)

con = sqlite3.connect("../data/my_data1.db")
df = pd.read_csv(CSV)
df.to_sql("SPACEXTBL", con, if_exists="replace", index=False)

# remove blank rows (rows with no date), like in the IBM lab
con.execute("DROP TABLE IF EXISTS SPACEXTABLE")
con.execute("CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL")
print("rows in SPACEXTABLE:", con.execute("SELECT COUNT(*) FROM SPACEXTABLE").fetchone()[0])

def sql(query):
    """Run a query, print it and its result."""
    print(query)
    print("-" * 60)
    result = pd.read_sql(query, con)
    print(result.to_string(index=False))
    return result

rows in SPACEXTABLE: 101


## 1. Unique launch sites

In [2]:
q1 = sql("""SELECT DISTINCT Launch_Site FROM SPACEXTABLE;""")

SELECT DISTINCT Launch_Site FROM SPACEXTABLE;
------------------------------------------------------------
 Launch_Site
 CCAFS LC-40
 VAFB SLC-4E
  KSC LC-39A
CCAFS SLC-40


## 2. Five records where the launch site starts with 'CCA'

In [3]:
q2 = sql("""SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5;""")

SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5;
------------------------------------------------------------
      Date Time (UTC) Booster_Version Launch_Site                                                       Payload  PAYLOAD_MASS__KG_     Orbit        Customer Mission_Outcome     Landing_Outcome
2010-06-04   18:45:00  F9 v1.0  B0003 CCAFS LC-40                          Dragon Spacecraft Qualification Unit                  0       LEO          SpaceX         Success Failure (parachute)
2010-12-08   15:43:00  F9 v1.0  B0004 CCAFS LC-40 Dragon demo flight C1, two CubeSats, barrel of Brouere cheese                  0 LEO (ISS) NASA (COTS) NRO         Success Failure (parachute)
2012-05-22    7:44:00  F9 v1.0  B0005 CCAFS LC-40                                         Dragon demo flight C2                525 LEO (ISS)     NASA (COTS)         Success          No attempt
2012-10-08    0:35:00  F9 v1.0  B0006 CCAFS LC-40                                                  Spa

## 3. Total payload mass carried for NASA (CRS)

In [4]:
q3 = sql("""SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_kg FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)';""")

SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_kg FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)';
------------------------------------------------------------
 Total_Payload_kg
            45596


## 4. Average payload mass carried by booster version F9 v1.1

In [5]:
q4 = sql("""SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_kg FROM SPACEXTABLE WHERE Booster_Version = 'F9 v1.1';""")

SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_kg FROM SPACEXTABLE WHERE Booster_Version = 'F9 v1.1';
------------------------------------------------------------
 Avg_Payload_kg
         2928.4


## 5. Date of the first successful landing on a ground pad

In [6]:
q5 = sql("""SELECT MIN(Date) AS First_Ground_Pad_Success FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)';""")

SELECT MIN(Date) AS First_Ground_Pad_Success FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)';
------------------------------------------------------------
First_Ground_Pad_Success
              2015-12-22


## 6. Boosters that landed on a drone ship with payload between 4000 and 6000 kg

In [7]:
q6 = sql("""SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000;""")

SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000;
------------------------------------------------------------
Booster_Version  PAYLOAD_MASS__KG_
    F9 FT B1022               4696
    F9 FT B1026               4600
 F9 FT  B1021.2               5300
 F9 FT  B1031.2               5200


## 7. Total number of successful and failed mission outcomes

In [8]:
q7 = sql("""SELECT Mission_Outcome, COUNT(*) AS Total FROM SPACEXTABLE
GROUP BY Mission_Outcome;""")

SELECT Mission_Outcome, COUNT(*) AS Total FROM SPACEXTABLE
GROUP BY Mission_Outcome;
------------------------------------------------------------
                 Mission_Outcome  Total
             Failure (in flight)      1
                         Success     98
                        Success       1
Success (payload status unclear)      1


## 7b. Same question, grouped into Success / Failure (one 'Success ' value has an extra space)

In [9]:
q8 = sql("""SELECT CASE WHEN Mission_Outcome LIKE 'Success%' THEN 'Success' ELSE 'Failure' END AS Outcome,
       COUNT(*) AS Total
FROM SPACEXTABLE GROUP BY Outcome;""")

SELECT CASE WHEN Mission_Outcome LIKE 'Success%' THEN 'Success' ELSE 'Failure' END AS Outcome,
       COUNT(*) AS Total
FROM SPACEXTABLE GROUP BY Outcome;
------------------------------------------------------------
Outcome  Total
Failure      1
Success    100


## 8. Boosters that carried the maximum payload mass (subquery)

In [10]:
q9 = sql("""SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE);""")

SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE);
------------------------------------------------------------
Booster_Version  PAYLOAD_MASS__KG_
  F9 B5 B1048.4              15600
  F9 B5 B1049.4              15600
  F9 B5 B1051.3              15600
  F9 B5 B1056.4              15600
  F9 B5 B1048.5              15600
  F9 B5 B1051.4              15600
  F9 B5 B1049.5              15600
 F9 B5 B1060.2               15600
 F9 B5 B1058.3               15600
  F9 B5 B1051.6              15600
  F9 B5 B1060.3              15600
 F9 B5 B1049.7               15600


## 9. Failed drone-ship landings in 2015, by month

In [11]:
q10 = sql("""SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTABLE
WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date, 1, 4) = '2015';""")

SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTABLE
WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date, 1, 4) = '2015';
------------------------------------------------------------
Month      Landing_Outcome Booster_Version Launch_Site
   01 Failure (drone ship)   F9 v1.1 B1012 CCAFS LC-40
   04 Failure (drone ship)   F9 v1.1 B1015 CCAFS LC-40


## 10. Rank landing outcomes between 2010-06-04 and 2017-03-20

In [12]:
q11 = sql("""SELECT Landing_Outcome, COUNT(*) AS Total FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Total DESC;""")

SELECT Landing_Outcome, COUNT(*) AS Total FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Total DESC;
------------------------------------------------------------
       Landing_Outcome  Total
            No attempt     10
  Success (drone ship)      5
  Failure (drone ship)      5
  Success (ground pad)      3
    Controlled (ocean)      3
  Uncontrolled (ocean)      2
   Failure (parachute)      2
Precluded (drone ship)      1


## Extra: landing success vs. failure counts (all years)

In [13]:
extra = sql("""SELECT CASE WHEN Landing_Outcome LIKE 'Success%' THEN 'Success'
            WHEN Landing_Outcome LIKE 'Failure%' THEN 'Failure'
            ELSE 'No attempt / other' END AS Landing_Result,
       COUNT(*) AS Total
FROM SPACEXTABLE GROUP BY Landing_Result ORDER BY Total DESC;""")

SELECT CASE WHEN Landing_Outcome LIKE 'Success%' THEN 'Success'
            WHEN Landing_Outcome LIKE 'Failure%' THEN 'Failure'
            ELSE 'No attempt / other' END AS Landing_Result,
       COUNT(*) AS Total
FROM SPACEXTABLE GROUP BY Landing_Result ORDER BY Total DESC;
------------------------------------------------------------
    Landing_Result  Total
           Success     61
No attempt / other     30
           Failure     10


In [14]:
con.close()